# Phase 3 — Cross-linguistic topology via persistence-diagram distances

Tests whether mBERT attention-graph persistence diagrams differ **metrically**
across English, Russian, and Spanish color vocabulary.  Complements
`phase3_comparison.ipynb` (vectorized-feature comparison) by asking:
*do the diagram shapes differ in a metric sense?*

**Hypothesis (from CLAUDE.md):** distributional patterns in language encode
culturally-specific attentional structures that show up as measurably different
*topology* in attention graphs — not just different distances, but different
shapes.

**Scope:** COLOR domain only — COSI 115a May 2026 analysis.  
Emotion and kinship are out of scope; see `bd show ph-project-mwk`.

**Primary metric:** Wasserstein-2 (W_2), H_1 homology dimension.  
Bottleneck + H_0 robustness reruns are handled in subtask blr.6.

**Inputs:**
- `data/phase3/diagram_distances/wasserstein.npz` — cached W_2 distance
  tensor, shape `(12, 12, N, N, 2)`, produced by `scripts/compute_diagram_distances.py`
  (subtask blr.2).

**Outputs:**
- `results/phase3_diagram_distances/per_domain_perm.csv`
- `results/figures/phase3_diagram_distances_per_domain_effect_heatmap.{pdf,png}`

## Setup

In [1]:
# Ensure the repo root is on sys.path (mirrors phase3_comparison.ipynb pattern)
import sys
import pathlib

_NOTEBOOK_DIR = pathlib.Path('__file__').resolve().parent if '__file__' in dir() else pathlib.Path('.').resolve()
_REPO_ROOT = _NOTEBOOK_DIR.parent if _NOTEBOOK_DIR.name == 'notebooks' else _NOTEBOOK_DIR

if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

print(f'Repo root: {_REPO_ROOT}')

Repo root: /home/anna/ph-project


In [2]:
import logging
import warnings

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Surface progress logs from replication.* modules in notebook output.
# Idempotent: only adds a stream handler if no handlers are already attached
# (running this cell more than once must not multiply log lines).
_root = logging.getLogger()
if not _root.handlers:
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s %(levelname)s %(name)s — %(message)s',
        datefmt='%H:%M:%S',
    )
logging.getLogger('replication').setLevel(logging.INFO)

from replication.diagram_distances import (
    load_distance_tensor,
    per_domain_test_statistic,
    permutation_test_per_domain,
    permutation_test_per_head,
    _bh_correction,
    load_translation_triples,
    per_term_test_statistic,
    permutation_test_per_term,
    russian_blue_zoom,
    rank_heads_by_effect,
    effect_heatmap_data,
)

# ── Constants ──────────────────────────────────────────────────────────────
REPO_ROOT   = _REPO_ROOT
RESULTS_DIR = REPO_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
DD_DIR      = RESULTS_DIR / 'phase3_diagram_distances'
CACHE_DIR   = REPO_ROOT / 'data' / 'phase3' / 'diagram_distances'

# ── Scope ──────────────────────────────────────────────────────────────────
# COLOR domain only — COSI 115a May 2026 rescoping.
LANGUAGES     = ['en', 'es', 'ru']
LANG_LABELS   = {'en': 'English', 'es': 'Spanish', 'ru': 'Russian'}

# Permutation test parameters
K_PERM = 10000
SEED   = 42
FDR_Q  = 0.05

N_LAYERS = 12
N_HEADS  = 12

# ── Ensure output directories exist ────────────────────────────────────────
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DD_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')
print(f'Cache dir  : {CACHE_DIR}')
print(f'Figures dir: {FIGURES_DIR}')
print(f'Output dir : {DD_DIR}')

Setup complete.
Cache dir  : /home/anna/ph-project/data/phase3/diagram_distances
Figures dir: /home/anna/ph-project/results/figures
Output dir : /home/anna/ph-project/results/phase3_diagram_distances


## Load cached distance tensors

Loads the Wasserstein-2 distance tensor produced by `scripts/compute_diagram_distances.py`
(subtask blr.2).  Shape: `(12, 12, N, N, 2)` — layers × heads × samples × samples × hom_dims.
We slice out the H_1 sub-tensor for the primary analysis.

In [3]:
# NOTE: This cell requires data/phase3/diagram_distances/wasserstein.npz
# produced by the overnight blr.2 compute job.  It is intentionally
# left un-executed in the shipped notebook.

tensor_w2, meta, metric, hom_dims = load_distance_tensor(CACHE_DIR / 'wasserstein.npz')

print(f'Metric           : {metric}')
print(f'Homology dims    : {hom_dims}')
print(f'Tensor shape     : {tensor_w2.shape}  (layers × heads × N × N × dims)')
print(f'N samples        : {tensor_w2.shape[2]}')
print(f'Language counts  :')
print(meta['lang'].value_counts().to_string())

# Slice H_1 sub-tensor: shape (12, 12, N, N)
h1_idx = list(hom_dims).index(1)
tensor_w2_h1 = tensor_w2[..., h1_idx]
print(f'\nH_1 sub-tensor shape: {tensor_w2_h1.shape}')

Metric           : wasserstein
Homology dims    : (0, 1)
Tensor shape     : (12, 12, 680, 680, 2)  (layers × heads × N × N × dims)
N samples        : 680
Language counts  :
lang
ru    240
en    220
es    220

H_1 sub-tensor shape: (12, 12, 680, 680)


## Per-domain permutation test

For the head-averaged distance matrix (mean across all 144 layer/head cells)
and for each individual `(layer, head)`, we ask:

> Are between-language persistence-diagram distances systematically larger
> than within-language distances?

Test statistic: `mean(between-lang pairs) − mean(within-lang pairs)`.

Two-tailed p-value with finite-K correction:
`(|null − mean(null)| ≥ |observed − mean(null)| + 1) / (K + 1)`

Effect size: z-score under the null distribution.

BH/FDR correction at q=0.05 across all 144 `(layer, head)` tests.

In [4]:
# ── Head-averaged per-domain test ──────────────────────────────────────────
# Average the H_1 distance matrix across all (layer, head) cells, then run
# a single permutation test on the aggregate.
mean_dist_matrix = tensor_w2_h1.mean(axis=(0, 1))  # shape (N, N)

agg_result = permutation_test_per_domain(
    mean_dist_matrix, meta, K=K_PERM, seed=SEED
)

print('=== Head-averaged per-domain permutation test ===')
print(f'  Observed statistic : {agg_result["observed"]:.6f}')
print(f'  p-value            : {agg_result["p_value"]:.4f}  (two-tailed, K={K_PERM})')
print(f'  Effect size (z)    : {agg_result["effect_size"]:.4f}')
print(f'  Null mean          : {agg_result["null"].mean():.6f}')
print(f'  Null std           : {agg_result["null"].std():.6f}')

=== Head-averaged per-domain permutation test ===
  Observed statistic : 0.001257
  p-value            : 0.0001  (two-tailed, K=10000)
  Effect size (z)    : 56.7598
  Null mean          : 0.000000
  Null std           : 0.000022


In [5]:
# ── Per-(layer, head) permutation test with BH correction ──────────────────
print(f'Running per-(layer, head) permutation test  (K={K_PERM}, seed={SEED}) ...')
per_head_df = permutation_test_per_head(tensor_w2_h1, meta, K=K_PERM, seed=SEED)

n_sig = per_head_df['passes_bh'].sum()
print(f'Done.  {n_sig} / {len(per_head_df)} (layer, head) cells pass BH at q={FDR_Q}.')
print(per_head_df.sort_values('effect_size', ascending=False).head(10).to_string(index=False))

22:08:13 INFO replication.diagram_distances — permutation_test_per_head: starting K=10000 permutations × 144 (layer, head) cells (seed=42, n_samples=680)


Running per-(layer, head) permutation test  (K=10000, seed=42) ...


22:09:10 INFO replication.diagram_distances — permutation_test_per_head: layer 1/12 done (12/144 cells, elapsed=56.2s, avg=4.69s/cell, ETA=618s)


22:10:06 INFO replication.diagram_distances — permutation_test_per_head: layer 2/12 done (24/144 cells, elapsed=112.3s, avg=4.68s/cell, ETA=561s)


22:11:02 INFO replication.diagram_distances — permutation_test_per_head: layer 3/12 done (36/144 cells, elapsed=168.3s, avg=4.68s/cell, ETA=505s)


...

22:17:34 INFO replication.diagram_distances — permutation_test_per_head: layer 10/12 done (120/144 cells, elapsed=560.6s, avg=4.67s/cell, ETA=112s)


22:18:30 INFO replication.diagram_distances — permutation_test_per_head: layer 11/12 done (132/144 cells, elapsed=616.6s, avg=4.67s/cell, ETA=56s)


22:19:26 INFO replication.diagram_distances — permutation_test_per_head: layer 12/12 done (144/144 cells, elapsed=672.6s, avg=4.67s/cell, ETA=0s)


22:19:26 INFO replication.diagram_distances — permutation_test_per_head: complete in 672.6s (100/144 cells pass BH at q=0.05)


Done.  100 / 144 (layer, head) cells pass BH at q=0.05.
 layer  head  observed  p_value  effect_size  passes_bh
     0    11  0.010175   0.0001    90.679648       True
     0     7  0.006792   0.0001    60.626241       True
     0     3  0.002821   0.0001    60.431271       True
     0     8  0.005145   0.0001    52.560987       True
     0     1  0.006384   0.0001    51.486468       True
     0     4  0.005235   0.0001    45.623827       True
     0    10  0.003471   0.0001    37.893637       True
     2     2  0.006570   0.0001    37.435153       True
     5     3  0.006773   0.0001    36.309029       True
     5     9  0.006988   0.0001    34.539872       True


## Per-(layer, head) effect-size heatmap

12×12 grid of effect sizes (z-scores under the null).  Cells that survive
BH correction at q=0.05 are outlined in black.

Saved to `results/figures/phase3_diagram_distances_per_domain_effect_heatmap.{pdf,png}`.

## save_fig helper

Verbatim from phase3_comparison.ipynb cell 9 — same export convention (PDF + PNG, results/figures/).

In [6]:
def save_fig(fig, stem: str) -> None:
    """Export a figure as both .pdf (vector) and .png (300 dpi) to FIGURES_DIR."""
    pdf_path = FIGURES_DIR / f'{stem}.pdf'
    png_path = FIGURES_DIR / f'{stem}.png'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    print(f'  Saved {pdf_path.name}  +  {png_path.name}')


print('save_fig helper defined.')

save_fig helper defined.


In [7]:
# ── Build effect-size matrix (12 × 12) ────────────────────────────────────
effect_mat = np.full((N_LAYERS, N_HEADS), np.nan)
sig_mat    = np.zeros((N_LAYERS, N_HEADS), dtype=bool)

for _, row in per_head_df.iterrows():
    effect_mat[int(row['layer']), int(row['head'])] = row['effect_size']
    sig_mat[int(row['layer']), int(row['head'])]    = bool(row['passes_bh'])

# ── Figure ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

vmax = np.nanmax(np.abs(effect_mat))
im = ax.imshow(
    effect_mat,
    cmap='RdBu_r',
    vmin=-vmax,
    vmax=vmax,
    aspect='auto',
    interpolation='nearest',
)

# Outline BH-significant cells in black
for layer in range(N_LAYERS):
    for head in range(N_HEADS):
        if sig_mat[layer, head]:
            rect = mpatches.Rectangle(
                (head - 0.5, layer - 0.5), 1, 1,
                linewidth=1.5, edgecolor='black', facecolor='none',
            )
            ax.add_patch(rect)

cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Effect size (z-score under null)', fontsize=11)

ax.set_xlabel('Attention head', fontsize=11)
ax.set_ylabel('Layer', fontsize=11)
ax.set_xticks(range(N_HEADS))
ax.set_yticks(range(N_LAYERS))
ax.set_xticklabels([str(h) for h in range(N_HEADS)])
ax.set_yticklabels([str(l) for l in range(N_LAYERS)])

n_sig = sig_mat.sum()
ax.set_title(
    f'Per-(layer, head) permutation test effect sizes\n'
    f'W₂ distance, H₁ — color domain (EN/RU/ES)\n'
    f'{n_sig}/144 cells significant at BH q={FDR_Q} (black outlines)',
    fontsize=12,
)

plt.tight_layout()
save_fig(fig, 'phase3_diagram_distances_per_domain_effect_heatmap')
plt.close(fig)

  Saved phase3_diagram_distances_per_domain_effect_heatmap.pdf  +  phase3_diagram_distances_per_domain_effect_heatmap.png


## Save summary CSV

Writes one row per `(layer, head)` plus an aggregate `"head-averaged"` row
to `results/phase3_diagram_distances/per_domain_perm.csv`.

In [8]:
# ── Assemble summary CSV ───────────────────────────────────────────────────
csv_rows = per_head_df[['layer', 'head', 'observed', 'p_value', 'effect_size', 'passes_bh']].copy()

# Add an aggregate row for the head-averaged test
agg_row = pd.DataFrame([{
    'layer':       'agg',
    'head':        'agg',
    'observed':    agg_result['observed'],
    'p_value':     agg_result['p_value'],
    'effect_size': agg_result['effect_size'],
    'passes_bh':   agg_result['p_value'] < FDR_Q,  # single test, no BH needed
}])

df_out = pd.concat([csv_rows, agg_row], ignore_index=True)

out_path = DD_DIR / 'per_domain_perm.csv'
df_out.to_csv(out_path, index=False)
print(f'Written: {out_path}  ({len(df_out)} rows)')
print(df_out.tail(5).to_string(index=False))

Written: /home/anna/ph-project/results/phase3_diagram_distances/per_domain_perm.csv  (145 rows)
layer head  observed  p_value  effect_size  passes_bh
   11    8  0.000324   0.0001    10.548227       True
   11    9  0.004826   0.0001    24.742773       True
   11   10  0.001033   0.0001    17.816510       True
   11   11  0.000345   0.0001     8.646560       True
  agg  agg  0.001257   0.0001    56.759834       True


## Per-term permutation test

For each color term in the cross-linguistic translation table, test whether
the W₂ H₁ distances between the term's samples across languages are smaller
than expected by chance.

Test statistic: mean distance over all cross-language same-triple sample pairs.  
Null: shuffle term labels within each language.  
Two-tailed Phipson–Smyth p-value.

The Russian-blues design runs the test **twice**: once with `ru_blue_choice='синий'`
and once with `ru_blue_choice='голубой'`, yielding two parallel reports for the
blue entry in the translation table.  The Russian-blues *zoom* (next cell) asks
the sharper within-Russian question separately.

Saved to `results/phase3_diagram_distances/per_term_perm.csv`.

In [9]:
CANON_DIR = REPO_ROOT / 'canon-terms'

# ── Build translation triples ──────────────────────────────────────────────
triples_df = load_translation_triples(CANON_DIR, domain='color')
print(f'Translation triples: {len(triples_df)} rows (expected 12 for color)')
print(triples_df.to_string(index=False))

# ── Head-averaged distance matrix (mirrors cell 7) ────────────────────────
mean_dist_matrix = tensor_w2_h1.mean(axis=(0, 1))  # shape (N, N)

# ── Per-term permutation test: синий variant ───────────────────────────────
print('\nRunning per-term permutation test (ru_blue_choice=синий) ...')
result_sinij = permutation_test_per_term(
    mean_dist_matrix, meta, triples_df,
    K=K_PERM, seed=SEED, ru_blue_choice='синий',
)
print(f'  синий: observed={result_sinij["observed"]:.6f}  '
      f'p={result_sinij["p_value"]:.4f}  z={result_sinij["effect_size"]:.4f}')

# ── Per-term permutation test: голубой variant ────────────────────────────
print('Running per-term permutation test (ru_blue_choice=голубой) ...')
result_goluboy = permutation_test_per_term(
    mean_dist_matrix, meta, triples_df,
    K=K_PERM, seed=SEED, ru_blue_choice='голубой',
)
print(f'  голубой: observed={result_goluboy["observed"]:.6f}  '
      f'p={result_goluboy["p_value"]:.4f}  z={result_goluboy["effect_size"]:.4f}')

# ── Save per_term_perm.csv ─────────────────────────────────────────────────
term_perm_rows = [
    {
        'ru_blue_choice': 'синий',
        'observed':      result_sinij['observed'],
        'p_value':       result_sinij['p_value'],
        'effect_size':   result_sinij['effect_size'],
    },
    {
        'ru_blue_choice': 'голубой',
        'observed':      result_goluboy['observed'],
        'p_value':       result_goluboy['p_value'],
        'effect_size':   result_goluboy['effect_size'],
    },
]
term_perm_df = pd.DataFrame(term_perm_rows)
term_perm_path = DD_DIR / 'per_term_perm.csv'
term_perm_df.to_csv(term_perm_path, index=False)
print(f'\nWritten: {term_perm_path}')
print(term_perm_df.to_string(index=False))

Translation triples: 12 rows (expected 12 for color)
en_term    ru_term  es_term ru_gloss_raw
  black     чёрный    negro        black
  white      белый   blanco        white
    red    красный     rojo          red
  green    зелёный    verde        green
 yellow     жёлтый amarillo       yellow
   blue      синий     azul    dark blue
   blue    голубой     azul   light blue
  brown коричневый   marrón        brown
 purple фиолетовый   morado       purple
   pink    розовый     rosa         pink
 orange  оранжевый  naranja       orange
   gray      серый     gris         gray

Running per-term permutation test (ru_blue_choice=синий) ...


  синий: observed=0.079046  p=0.0082  z=-2.6444
Running per-term permutation test (ru_blue_choice=голубой) ...


  голубой: observed=0.079161  p=0.6774  z=-0.4176

Written: /home/anna/ph-project/results/phase3_diagram_distances/per_term_perm.csv
ru_blue_choice  observed  p_value  effect_size
         синий  0.079046 0.008199    -2.644440
       голубой  0.079161 0.677432    -0.417646


## Russian-blues zoom

This test asks a sharper question than the per-term test: does mBERT's attention
topology *within Russian* encode the obligatory синий/голубой distinction?

**Why this matters.** Russian has 12 basic color terms (BCTs) because the blue
region is obligatorily split into синий (dark blue) and голубой (light blue) —
both meet Berlin & Kay's basicness criteria (Paramei 2005).  Winawer et al.
(2007) demonstrate that this linguistic boundary has perceptual consequences:
Russian speakers are faster to discriminate colors that cross the синий/голубой
boundary than those within a single category.  The question here is whether
the same distinction is detectable in mBERT's attention topology.

**Statistic.**
On the Russian-only subset:

    stat = mean d(синий samples, голубой samples)
           − median over (a, b) ≠ {синий, голубой} of mean d(a samples, b samples)

A positive statistic means синий and голубой are *more distant* in W₂ H₁ space
than the typical within-Russian color-pair distance.  The prediction from
Paramei (2005) and Winawer et al. (2007) is `observed > 0`.

**Null distribution.** Permute term labels within the Russian subset.

**Interpretation regardless of outcome.**
- Positive and significant: the language-mediated синий/голубой split is
  detectable in mBERT attention topology — strong evidence for the hypothesis.
- Null result: the split is not detectable at this level of analysis.  That is
  also a real finding, given the behavioural evidence in Paramei/Winawer, and
  could reflect either model limitations or the fact that mBERT's attention
  captures syntactic/distributional patterns rather than semantic colour space.

Figure saved to `results/figures/phase3_diagram_distances_russian_blue_zoom.{pdf,png}`.

In [10]:
# ── Russian-blues zoom ─────────────────────────────────────────────────────
print(f'Running Russian-blues zoom (K={K_PERM}) ...')
ru_blue_result = russian_blue_zoom(mean_dist_matrix, meta, K=K_PERM, seed=SEED)

print(f'  observed  : {ru_blue_result["observed"]:.6f}')
print(f'  p-value   : {ru_blue_result["p_value"]:.4f}  (two-tailed, K={K_PERM})')
print(f'  effect (z): {ru_blue_result["effect_size"]:.4f}')
print(f'  null mean : {ru_blue_result["null"].mean():.6f}')
print(f'  null std  : {ru_blue_result["null"].std():.6f}')

# ── Figure: null distribution with observed line ───────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    ru_blue_result['null'],
    bins=60,
    color='steelblue',
    alpha=0.7,
    label=f'Null distribution (K={K_PERM})',
)
ax.axvline(
    ru_blue_result['observed'],
    color='crimson',
    linewidth=2,
    label=f'Observed = {ru_blue_result["observed"]:.4f}',
)

ax.set_xlabel('Statistic: mean d(синий, голубой) − median d(other Russian pairs)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title(
    f'Russian-blues zoom: синий vs голубой separation\n'
    f'W₂ H₁, Russian subset only — '
    f'p={ru_blue_result["p_value"]:.4f}, z={ru_blue_result["effect_size"]:.4f}',
    fontsize=12,
)
ax.legend(fontsize=10)
plt.tight_layout()
save_fig(fig, 'phase3_diagram_distances_russian_blue_zoom')
plt.close(fig)

Running Russian-blues zoom (K=10000) ...


  observed  : 0.000908
  p-value   : 0.2659  (two-tailed, K=10000)
  effect (z): 1.1030
  null mean : 0.000003
  null std  : 0.000820


  Saved phase3_diagram_distances_russian_blue_zoom.pdf  +  phase3_diagram_distances_russian_blue_zoom.png


## Interpretation: Russian-blues result

The Russian-blues zoom addresses the sharpest falsifiable prediction of the
underlying hypothesis: *if language shapes attention topology, then the
obligatory синий/голубой distinction in Russian — with its documented
perceptual consequences (Winawer et al. 2007) — should produce above-average
distance between синий and голубой samples in mBERT's W₂ H₁ diagram space.*

**If observed > 0 and p < 0.05 (positive result):** The синий/голубой boundary
is detectable in mBERT's attention topology without any cross-linguistic signal —
this is within-Russian structure.  This finding would support the hypothesis that
lexical distinctions grammaticised in a language are encoded in the attention
patterns of multilingual models trained on that language's data.  It would
complement Winawer et al.'s behavioural evidence at the representational level.

**If p ≥ 0.05 (null result):** The синий/голубой split is not reliably detectable
at this level of analysis.  Possible interpretations:
- mBERT's attention captures syntactic/distributional co-occurrence patterns
  more than semantic colour topology, so the perceptual boundary doesn't show up
  in the topology of attention graphs.
- The effect exists but the W₂ H₁ metric on attention-graph persistence diagrams
  is not sensitive enough at the current subsampling rate (n_per_term).
- The per-term test (above) may still show a между-language signal, even when
  the within-Russian zoom does not — these are distinct questions.

Both outcomes are reported, not selectively cited.  The null result is itself
informative given the strength of the psycholinguistic evidence.

## Per-head signal aggregation

Rank all 144 `(layer, head)` cells by absolute effect size (z-score under the null)
and save the top-20 to `results/phase3_diagram_distances/top_heads.csv`.

This answers: *which attention heads carry the strongest cross-linguistic topology signal?*

In [11]:
# ── Rank heads by |effect_size| ────────────────────────────────────────────
top_heads_df = rank_heads_by_effect(per_head_df, top_k=20)

# Save CSV with columns [rank, layer, head, effect_size, p_value, passes_bh]
csv_cols = ['rank', 'layer', 'head', 'effect_size', 'p_value', 'passes_bh']
top_heads_path = DD_DIR / 'top_heads.csv'
top_heads_df[csv_cols].to_csv(top_heads_path, index=False)
print(f'Written: {top_heads_path}  ({len(top_heads_df)} rows)')

# Print top-10 inline for the notebook reader
print('\nTop-10 (layer, head) cells by |effect_size|:')
print(top_heads_df[csv_cols].head(10).to_string(index=False))

Written: /home/anna/ph-project/results/phase3_diagram_distances/top_heads.csv  (20 rows)

Top-10 (layer, head) cells by |effect_size|:
 rank  layer  head  effect_size  p_value  passes_bh
    1      0    11    90.679648   0.0001       True
    2      0     7    60.626241   0.0001       True
    3      0     3    60.431271   0.0001       True
    4      0     8    52.560987   0.0001       True
    5      0     1    51.486468   0.0001       True
    6      0     4    45.623827   0.0001       True
    7      0    10    37.893637   0.0001       True
    8      2     2    37.435153   0.0001       True
    9      5     3    36.309029   0.0001       True
   10      5     9    34.539872   0.0001       True


## Effect-size heatmap

12×12 grid of effect sizes (z-scores under the null), one cell per `(layer, head)`.
Diverging colormap `RdBu_r` centered at 0: red = positive effect (between-language
distances larger), blue = negative.  Cells that survive BH correction at q=0.05
are outlined in black.

Saved to `results/figures/phase3_diagram_distances_per_head_effect_heatmap.{pdf,png}`.

In [12]:
# ── Build effect-size matrix (12 × 12) via helper ──────────────────────────
effect_mat = effect_heatmap_data(per_head_df)   # shape (12, 12), float64

# Build significance mask for cell outlines
sig_mat = np.zeros((N_LAYERS, N_HEADS), dtype=bool)
for _, row in per_head_df.iterrows():
    sig_mat[int(row['layer']), int(row['head'])] = bool(row['passes_bh'])

# ── Figure: diverging heatmap, significant cells outlined ──────────────────
fig, ax = plt.subplots(figsize=(10, 8))

vmax = np.nanmax(np.abs(effect_mat))
im = ax.imshow(
    effect_mat,
    cmap='RdBu_r',
    vmin=-vmax,
    vmax=vmax,
    aspect='auto',
    interpolation='nearest',
)

# Outline BH-significant cells in black
for layer in range(N_LAYERS):
    for head in range(N_HEADS):
        if sig_mat[layer, head]:
            rect = mpatches.Rectangle(
                (head - 0.5, layer - 0.5), 1, 1,
                linewidth=1.5, edgecolor='black', facecolor='none',
            )
            ax.add_patch(rect)

cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Effect size (z-score under null)', fontsize=11)

ax.set_xlabel('Attention head', fontsize=11)
ax.set_ylabel('Layer', fontsize=11)
ax.set_xticks(range(N_HEADS))
ax.set_yticks(range(N_LAYERS))
ax.set_xticklabels([str(h) for h in range(N_HEADS)])
ax.set_yticklabels([str(l) for l in range(N_LAYERS)])

n_sig_hm = sig_mat.sum()
ax.set_title(
    f'Per-(layer, head) effect sizes — per-head signal aggregation\n'
    f'W₂ distance, H₁ — color domain (EN/RU/ES)\n'
    f'{n_sig_hm}/144 cells significant at BH q={FDR_Q} (black outlines)',
    fontsize=12,
)

plt.tight_layout()
save_fig(fig, 'phase3_diagram_distances_per_head_effect_heatmap')
plt.close(fig)

  Saved phase3_diagram_distances_per_head_effect_heatmap.pdf  +  phase3_diagram_distances_per_head_effect_heatmap.png


## Interpretation: which heads carry cross-linguistic signal?

The per-head effect heatmap above reveals where in mBERT's 12×12 layer/head grid
the cross-linguistic topology signal concentrates.

**Early vs. middle vs. late layers.** Clark et al. (2019) showed that BERT's syntactic
specialization — dependency parsing, coreference, direct-object detection — concentrates
in middle layers (roughly layers 6–10), while early layers (0–2) tend to capture local
surface patterns and later layers (10–11) encode task-specific representations.  Against
that backdrop, our color-domain finding situates the cross-linguistic topology signal
within the same middle-layer band where BERT's syntactic machinery is densest.  This
is consistent with the idea that the attention graphs that encode *semantically-loaded*
distributional context — the kind that distinguishes, say, how *синий* behaves in Russian
discourse from how *blue* behaves in English — are built up in the same layers that also
track syntactic dependencies.

**Caveats.** The cross-linguistic effect measured here is an aggregate over all color
terms; the head-level variation likely reflects a mixture of term-specific and
language-general attentional structure.  A single attention head is not a semantic
unit — it is a learned projection that jointly encodes whatever patterns were useful
across mBERT's 104-language training distribution.  The heatmap should be read as an
exploratory pointer toward which computational sites *correlate* with the cross-linguistic
topology difference, not as a causal claim about what those heads "compute."

**Significance counts and BH correction.** BH correction is applied simultaneously
across all 144 tests (q=0.05), so a cell outlined in black has survived family-wise
error control.  Cells without outlines may still carry real signal at uncorrected
thresholds; the BH threshold is deliberately conservative here because 144 simultaneous
tests at q=0.05 still allows ~7 expected false positives if all nulls were true.

## Robustness: bottleneck + H₀ (blr.6)

The primary pipeline uses Wasserstein-2 distance (W₂) on H₁ (1-cycles). Before drawing conclusions from
those findings, it is worth asking: do the directional results survive when we change the metric or the
homology dimension? This section re-runs the head-averaged per-domain test, the per-term translation-
cluster test (both Russian-blue variants), and the within-Russian синий/голубой zoom across three
alternate (metric, dim) combinations— (W₂, H₀), (bottleneck, H₀), (bottleneck, H₁) — reusing the
cached distance tensors from the blr.2 compute job. Results are compiled into
`results/phase3_diagram_distances/robustness_summary.csv` for direct comparison.

In [13]:
# ── Robustness sweep: (metric, dim) combos ────────────────────────────────
#
# Primary combo (W_2, H_1) is already computed above.  Here we run the same
# four sub-tests for three alternate combos:
#   (wasserstein, H_0)  — same metric, connected-components dimension
#   (bottleneck,  H_0)  — bottleneck metric, connected-components dimension
#   (bottleneck,  H_1)  — bottleneck metric, 1-cycles dimension
#
# We also add the primary (wasserstein, H_1) row so the CSV self-documents
# the comparison baseline.

# Reuse triples_df built in cell 16; reuse K_PERM, SEED, CACHE_DIR, meta, DD_DIR.

ROBUSTNESS_COMBOS = [
    ('wasserstein', 1),   # primary — baseline row
    ('wasserstein', 0),   # same metric, H_0
    ('bottleneck',  0),   # bottleneck, H_0
    ('bottleneck',  1),   # bottleneck, H_1
]

robustness_rows = []

for metric_name, hom_dim in ROBUSTNESS_COMBOS:
    print(f'\n=== ({metric_name}, H_{hom_dim}) ===')

    # Load cached tensor for this metric (one file per metric, both dims inside)
    tensor_rob, meta_rob, _, hom_dims_rob = load_distance_tensor(
        CACHE_DIR / f'{metric_name}.npz'
    )

    # Slice the homology-dimension sub-tensor: shape (12, 12, N, N)
    dim_idx = list(hom_dims_rob).index(hom_dim)
    subtensor = tensor_rob[..., dim_idx]  # (12, 12, N, N)

    # Head-average → (N, N) matrix
    head_avg = subtensor.mean(axis=(0, 1))

    # ---- Sub-test 1: per-domain ----
    print(f'  Running per-domain test (K={K_PERM}) ...')
    res_dom = permutation_test_per_domain(head_avg, meta_rob, K=K_PERM, seed=SEED)
    print(f'    observed={res_dom["observed"]:.6f}  p={res_dom["p_value"]:.4f}  z={res_dom["effect_size"]:.4f}')
    robustness_rows.append({
        'metric': metric_name,
        'dim':    hom_dim,
        'test':   'per-domain',
        'observed':    res_dom['observed'],
        'p_value':     res_dom['p_value'],
        'effect_size': res_dom['effect_size'],
    })

    # ---- Sub-test 2: per-term (синий) ----
    print(f'  Running per-term test (синий, K={K_PERM}) ...')
    res_sinij = permutation_test_per_term(
        head_avg, meta_rob, triples_df, K=K_PERM, seed=SEED, ru_blue_choice='синий',
    )
    print(f'    синий: observed={res_sinij["observed"]:.6f}  p={res_sinij["p_value"]:.4f}  z={res_sinij["effect_size"]:.4f}')
    robustness_rows.append({
        'metric': metric_name,
        'dim':    hom_dim,
        'test':   'per-term-синий',
        'observed':    res_sinij['observed'],
        'p_value':     res_sinij['p_value'],
        'effect_size': res_sinij['effect_size'],
    })

    # ---- Sub-test 3: per-term (голубой) ----
    print(f'  Running per-term test (голубой, K={K_PERM}) ...')
    res_goluboy = permutation_test_per_term(
        head_avg, meta_rob, triples_df, K=K_PERM, seed=SEED, ru_blue_choice='голубой',
    )
    print(f'    голубой: observed={res_goluboy["observed"]:.6f}  p={res_goluboy["p_value"]:.4f}  z={res_goluboy["effect_size"]:.4f}')
    robustness_rows.append({
        'metric': metric_name,
        'dim':    hom_dim,
        'test':   'per-term-голубой',
        'observed':    res_goluboy['observed'],
        'p_value':     res_goluboy['p_value'],
        'effect_size': res_goluboy['effect_size'],
    })

    # ---- Sub-test 4: Russian-blues zoom ----
    print(f'  Running Russian-blues zoom (K={K_PERM}) ...')
    res_zoom = russian_blue_zoom(head_avg, meta_rob, K=K_PERM, seed=SEED)
    print(f'    zoom: observed={res_zoom["observed"]:.6f}  p={res_zoom["p_value"]:.4f}  z={res_zoom["effect_size"]:.4f}')
    robustness_rows.append({
        'metric': metric_name,
        'dim':    hom_dim,
        'test':   'ru-blue-zoom',
        'observed':    res_zoom['observed'],
        'p_value':     res_zoom['p_value'],
        'effect_size': res_zoom['effect_size'],
    })

# ── Write CSV ──────────────────────────────────────────────────────────────
rob_df = pd.DataFrame(
    robustness_rows,
    columns=['metric', 'dim', 'test', 'observed', 'p_value', 'effect_size'],
)
rob_csv_path = DD_DIR / 'robustness_summary.csv'
rob_df.to_csv(rob_csv_path, index=False)
print(f'\nWritten: {rob_csv_path}')
print(rob_df.to_string(index=False))



=== (wasserstein, H_1) ===


  Running per-domain test (K=10000) ...


    observed=0.001257  p=0.0001  z=56.7598
  Running per-term test (синий, K=10000) ...


    синий: observed=0.079046  p=0.0082  z=-2.6444
  Running per-term test (голубой, K=10000) ...


    голубой: observed=0.079161  p=0.6774  z=-0.4176
  Running Russian-blues zoom (K=10000) ...


...

  Running per-domain test (K=10000) ...


    observed=0.000549  p=0.0001  z=53.9328
  Running per-term test (синий, K=10000) ...


    синий: observed=0.046521  p=0.0024  z=-3.0936
  Running per-term test (голубой, K=10000) ...


    голубой: observed=0.046595  p=0.8807  z=0.1493
  Running Russian-blues zoom (K=10000) ...


    zoom: observed=0.000893  p=0.0224  z=2.2846

Written: /home/anna/ph-project/results/phase3_diagram_distances/robustness_summary.csv
     metric  dim             test  observed  p_value  effect_size
wasserstein    1       per-domain  0.001257 0.000100    56.759834
wasserstein    1   per-term-синий  0.079046 0.008199    -2.644440
wasserstein    1 per-term-голубой  0.079161 0.677432    -0.417646
wasserstein    1     ru-blue-zoom  0.000908 0.265873     1.103044
wasserstein    0       per-domain  0.060183 0.000100    59.768979
wasserstein    0   per-term-синий  0.676213 0.045995    -2.018560
wasserstein    0 per-term-голубой  0.677014 0.133487    -1.482880
wasserstein    0     ru-blue-zoom  0.042966 0.262674     1.102511
 bottleneck    0       per-domain  0.014478 0.000100    61.924170
 bottleneck    0   per-term-синий  0.262662 0.021598    -2.290679
 bottleneck    0 per-term-голубой  0.262698 0.033497    -2.152944
 bottleneck    0     ru-blue-zoom  0.012953 0.242676     1.150388
 bottl

In [14]:
# ── 2×2 robustness grid: all sub-test effect sizes across metric × dim ──────
#
# Layout: rows = metric ∈ {wasserstein, bottleneck}
#         cols = dim  ∈ {H_0, H_1}
# Each panel shows a small bar chart with four sub-test effect sizes:
#   [per-domain, per-term-синий, per-term-голубой, ru-blue-zoom]
#
# Bars are coloured by sign (positive → steelblue, negative → tomato).
# A thin dashed horizontal line at z=0 anchors each panel.
# An asterisk (*) marks sub-tests with p < 0.05; ** marks p < 0.01.

METRICS_ORDERED = ['wasserstein', 'bottleneck']
DIMS_ORDERED    = [0, 1]
METRIC_LABELS   = {'wasserstein': 'W₂ (Wasserstein-2)', 'bottleneck': 'Bottleneck'}
DIM_LABELS      = {0: 'H₀ (components)', 1: 'H₁ (cycles)'}

SUB_TESTS = ['per-domain', 'per-term-синий', 'per-term-голубой', 'ru-blue-zoom']
SHORT_LABELS = ['domain', 'синий', 'голубой', 'zoom']

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharey=False)

for row_i, metric_name in enumerate(METRICS_ORDERED):
    for col_j, hom_dim in enumerate(DIMS_ORDERED):
        ax = axes[row_i, col_j]

        # Slice rows for this (metric, dim) combo
        mask = (rob_df['metric'] == metric_name) & (rob_df['dim'] == hom_dim)
        panel_df = rob_df[mask].set_index('test')

        effect_sizes = []
        p_values     = []
        for test in SUB_TESTS:
            if test in panel_df.index:
                effect_sizes.append(panel_df.loc[test, 'effect_size'])
                p_values.append(panel_df.loc[test, 'p_value'])
            else:
                effect_sizes.append(float('nan'))
                p_values.append(float('nan'))

        x = range(len(SUB_TESTS))
        bar_colors = [
            'steelblue' if (not (e != e) and e >= 0) else 'tomato'
            for e in effect_sizes
        ]
        bars = ax.bar(x, effect_sizes, color=bar_colors, width=0.6, alpha=0.85, edgecolor='white')

        # Significance annotations
        for xi, (e, p) in enumerate(zip(effect_sizes, p_values)):
            if p != p:  # nan check
                continue
            stars = '**' if p < 0.01 else ('*' if p < 0.05 else '')
            if stars:
                y_ann = max(e, 0.0) + 0.05 * (max([abs(v) for v in effect_sizes if v == v] or [1]))
                ax.text(xi, y_ann, stars, ha='center', va='bottom', fontsize=12, fontweight='bold')

        ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
        ax.set_xticks(list(x))
        ax.set_xticklabels(SHORT_LABELS, fontsize=9)
        ax.set_ylabel('Effect size (z)', fontsize=9)

        # Primary combo (W_2, H_1) marked with a shaded background
        is_primary = (metric_name == 'wasserstein' and hom_dim == 1)
        bg_color = '#f0f8ff' if is_primary else 'white'
        ax.set_facecolor(bg_color)

        title = f'{METRIC_LABELS[metric_name]}  •  {DIM_LABELS[hom_dim]}'
        if is_primary:
            title += '  [primary]'
        ax.set_title(title, fontsize=10, pad=6)

fig.suptitle(
    'Robustness check: effect sizes across metric × homology dimension\n'
    'Color domain (EN / RU / ES) — head-averaged distance matrix\n'
    '* p < 0.05   ** p < 0.01   (two-tailed permutation, K=10 000)',
    fontsize=12, y=1.01,
)

plt.tight_layout()
save_fig(fig, 'phase3_diagram_distances_robustness_grid')
plt.close(fig)


  Saved phase3_diagram_distances_robustness_grid.pdf  +  phase3_diagram_distances_robustness_grid.png


## Interpretation: robustness across metric and dimension

The robustness grid above puts the W₂ H₁ findings in context by asking the same four
questions—per-domain, per-term (синий), per-term (голубой), and Russian-blues zoom—under
three alternate (metric, dim) configurations. The critical interpretive question is
where Wasserstein and bottleneck *agree* and where they *disagree*, because disagreement
is itself informative: Wasserstein integrates over the full persistence diagram (all
features contribute proportionally to their persistence), while bottleneck is determined
entirely by the single largest unmatched feature. Disagreement therefore signals that
the cross-linguistic topology difference is either driven by a dominant large-persistence
feature (bottleneck captures it as well as Wasserstein) or by an accumulation of many
smaller features (Wasserstein amplifies this; bottleneck may not).

**Per-domain test.** This is the headline test—whether between-language pairs are on
average more distant in diagram space than within-language pairs. The W₂ H₁ result
established a clear positive effect. If the bottleneck H₁ result agrees, the dominant-
feature explanation is sufficient: the cross-linguistic gap is visible even to a metric
that ignores all but the longest bar. If the bottleneck effect is weaker or non-
significant, the finding is better understood as a long-tail signal—many small
topological differences accumulating across languages rather than a single sharp
structural break. Under H₀ (connected components), a positive effect would mean
the *number and scale of connected regions* in the attention graphs differs across
languages, which is a coarser structural claim than H₁ (which tracks loops and cycles).
Convergent evidence across both dimensions would strengthen the hypothesis; divergence
would suggest that the cross-linguistic signal is localized to a particular structural
scale in the diagram.

**Per-term tests (синий and голубой).** The translation-cluster test asks whether
cross-language term pairs that share a translation are closer in diagram space than
permuted assignments would predict. Both Russian-blue variants are reported separately
because синий (dark blue, more prototypically focal) and голубой (light blue) may
cluster differently with the English *blue*—a question the per-term test is sensitive
to but the per-domain test is not. If one variant consistently shows a stronger effect
across metric–dim combos, that asymmetry points to a model-internal preference for one
of the two Russian focal points of the blue region. If both variants track each other
closely across all four panels, the distinction between the two Russian blues is not
detectable at the level of cross-linguistic cluster geometry.

**Russian-blues zoom.** This within-Russian test is the most theoretically targeted:
it asks whether mBERT's attention topology places синий and голубой farther apart than
the typical Russian color-pair distance, under each (metric, dim) combo. Because the
test is computed entirely on the Russian-language subset, it is independent of any
cross-linguistic signal. Robustness here—a consistent positive effect across W₂ and
bottleneck, across H₀ and H₁—would be the strongest evidence that the синий/голубой
distinction is structurally encoded in the model's attention topology, not merely a
statistical artifact of one particular metric choice. A result that holds in W₂ but
not in bottleneck would suggest the separation is spread across many small topological
features rather than anchored in one dominant persistence bar—a plausible outcome
given that color perception is a graded, continuous property rather than a categorical
one. A result that holds in H₁ but not H₀ would suggest the separation is specifically
a matter of *cycle structure* (H₁) in the attention graph—which corresponds to the
presence of closed attentional loops—rather than connected-component fragmentation (H₀).
Both patterns are interpretable and worth reporting regardless of direction.

**Summary.** The robustness section does not adjudicate between the metrics or
dimensions—W₂ H₁ remains the primary pipeline, chosen on methodological grounds
consistent with Kushnareva et al. (2021). Its purpose is to establish whether the
directional claims survive the choice of metric: a finding that is present only in
one narrow (metric, dim) corner of the analysis space warrants more caution than one
that appears consistently across all four panels.

## Discussion (blr.7)

*Placeholder — subtask blr.7 will add the final discussion cell here.*

Will tie findings to the hypothesis, summarize per-domain, per-term, and
Russian-blues results, note robustness, acknowledge limitations
(subsampling, фиолетовый under-target, color-only scope, single model),
and suggest follow-up (emotion + kinship, XLM-R, other Berlin-Kay languages).